<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/04.gradient-descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 04: Gradient Descent - Intro to Neural Learning
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 4 transitions from making predictions to **learning from mistakes**. In Andrew Trask's "Predict, Compare, Learn" triad, this chapter focuses on:
- **Compare:** Measuring the gap between hypothesis $\hat{y}$ and true target $y$ via an error/loss function.
- **Learn:** Determining the exact mathematical adjustment to the weight that reduces this error.

We contrast heuristic optimization (**Hot and Cold Learning**) with analytical optimization (**Gradient Descent**), demonstrating why calculus and linear algebra provide an exponential efficiency advantage.

### Key Chapter Takeaways:
1. **The Error Metric (Squared Error):** $E = (\hat{y} - y)^2$. Squaring guarantees positive values, penalizes large discrepancies quadratically, and creates a smooth parabolic convex error bowl.
2. **Derivative as Sensitivity:** The derivative $\frac{dE}{dw} = 2(\hat{y} - y)x$ provides both the exact direction and proportional magnitude required to decrease error.
3. **The Learning Rate ($\alpha$):** A scaling hyperparameter that prevents gradient updates from overshooting and diverging.


---
## 2. Theoretical Deep-Dive

### 2.1 The Mathematics of Error: Why Squared Error?
Let the prediction be $\hat{y} = x \cdot w$ and ground-truth target be $y$.
The raw difference is:
$$\text{raw\_error} = \hat{y} - y$$
Why don't we use raw error directly?
- If the network predicts $+10$ on one sample and $-10$ on another, the average raw error is $\frac{10 + (-10)}{2} = 0$, falsely suggesting perfect performance.

Instead, we define **Squared Error**:
$$E(w) = (\hat{y} - y)^2 = (x \cdot w - y)^2$$

Properties of Squared Error:
1. **Always Non-Negative:** $(\hat{y} - y)^2 \ge 0$, with minimum $0$ when $\hat{y} = y$.
2. **Quadratic Penalty:** An error of $10$ yields loss $100$, whereas an error of $1$ yields loss $1$. It heavily prioritizes eliminating major blunders.
3. **Smooth Convex Curvature:** It forms a continuous parabola with a single global minimum.

### 2.2 Hot and Cold Learning vs. Gradient Descent
- **Hot and Cold Learning:** Perturbs the weight by a small step amount ($w + \epsilon$ and $w - \epsilon$), compares errors, and picks the lower one.
  - *Limitation:* For $D$ weights, there are $2^D$ direction combinations. In modern networks with millions of parameters, this brute-force approach is computationally intractable.
- **Gradient Descent:** Uses calculus to compute the exact instantaneous rate of change analytically:
  $$\frac{dE}{dw} = \frac{d}{dw}(x \cdot w - y)^2 = 2(x \cdot w - y) \cdot \frac{d}{dw}(x \cdot w - y) = 2(\hat{y} - y) \cdot x$$

In Andrew Trask's formulation:
$$\text{error} = (\text{pred} - \text{goal})^2$$
$$\text{direction\_and\_amount} = (\text{pred} - \text{goal}) \cdot \text{input} \quad (\text{termed } \text{weight\_delta})$$
The factor of $2$ is often absorbed into the learning rate $\alpha$.

### 2.3 The Anatomy of `weight_delta = (pred - goal) * input`
Notice the mechanical roles of the two multiplied terms:
1. **$(\text{pred} - \text{goal})$ (Pure Error):**
   - If $\text{pred} > \text{goal}$, this term is positive $\implies$ weight must decrease.
   - If $\text{pred} < \text{goal}$, this term is negative $\implies$ weight must increase.
2. **$\text{input}$ (Scaling and Direction Reversal):**
   - If $\text{input}$ is large, a small change in weight drastically shifts prediction; hence, the gradient must be large.
   - If $\text{input}$ is zero, changing the weight has zero effect on prediction; hence, the gradient is zero.
   - If $\text{input}$ is negative, the algebraic sign of weight effect flips, which the input multiplication automatically corrects.

### 2.4 The Mechanics of Divergence (Breaking Gradient Descent)
If the learning rate $\alpha$ is too large, the weight update overshoots the minimum and lands on the opposite wall of the error parabola at a higher altitude.
For input $x$, the update step is:
$$w_{t+1} = w_t - \alpha \cdot 2(w_t x - y)x = w_t (1 - 2\alpha x^2) + 2\alpha x y$$
The recurrence converges if and only if:
$$|1 - 2\alpha x^2| < 1 \iff 0 < \alpha x^2 < 1 \iff \alpha < \frac{1}{x^2}$$
If $\alpha \ge \frac{1}{x^2}$, the system oscillates with expanding amplitude and **diverges to infinity**.


---
## 3. Code Reproduction: Comparing & Learning Algorithms

### 3.1 Step 2: Compare - Does Our Network Make Good Predictions?


In [1]:
knob_weight = 0.5
input_val = 0.5
goal_pred = 0.8

pred = input_val * knob_weight
error = (pred - goal_pred) ** 2

print(f"Prediction: {pred:.4f}")
print(f"Goal: {goal_pred:.4f}")
print(f"Squared Error: {error:.4f}")

Prediction: 0.2500
Goal: 0.8000
Squared Error: 0.3025

### 3.2 Hot and Cold Learning Implementation
Here we observe the perturbation-based method adjusting weight incrementally:


In [1]:
weight = 0.5
input_val = 0.5
goal_prediction = 0.8
step_amount = 0.001

print("--- Starting Hot and Cold Learning ---")
for iteration in range(1101):
    prediction = input_val * weight
    error = (prediction - goal_prediction) ** 2
    
    if iteration % 200 == 0:
        print(f"Iteration {iteration:4d} | Error: {error:.6f} | Prediction: {prediction:.4f} | Weight: {weight:.4f}")
        
    # Evaluate higher weight
    up_prediction = input_val * (weight + step_amount)
    up_error = (up_prediction - goal_prediction) ** 2
    
    # Evaluate lower weight
    down_prediction = input_val * (weight - step_amount)
    down_error = (down_prediction - goal_prediction) ** 2
    
    if down_error < up_error:
        weight -= step_amount
    if down_error > up_error:
        weight += step_amount

print(f"Final Weight: {weight:.4f}, Final Prediction: {input_val * weight:.4f}")

--- Starting Hot and Cold Learning ---
Iteration    0 | Error: 0.302500 | Prediction: 0.2500 | Weight: 0.5000
Iteration  200 | Error: 0.160000 | Prediction: 0.4000 | Weight: 0.8000
Iteration  400 | Error: 0.062500 | Prediction: 0.5500 | Weight: 1.1000
Iteration  600 | Error: 0.010000 | Prediction: 0.7000 | Weight: 1.4000
Iteration  800 | Error: 0.000000 | Prediction: 0.8000 | Weight: 1.6000
Iteration 1000 | Error: 0.000000 | Prediction: 0.8000 | Weight: 1.6000
Final Weight: 1.6000, Final Prediction: 0.8000

### 3.3 Gradient Descent: Direction and Amount from Error
Instead of trial-and-error perturbations, we compute the exact derivative:


In [1]:
weight = 0.5
goal_pred = 0.8
input_val = 0.5

print("--- Starting Analytical Gradient Descent ---")
for iteration in range(20):
    pred = input_val * weight
    error = (pred - goal_pred) ** 2
    direction_and_amount = (pred - goal_pred) * input_val
    weight = weight - direction_and_amount
    
    if iteration % 4 == 0 or iteration == 19:
        print(f"Iteration {iteration:2d} | Error: {error:.6f} | Prediction: {pred:.4f} | Delta: {direction_and_amount:.4f} | New Weight: {weight:.4f}")

--- Starting Analytical Gradient Descent ---
Iteration  0 | Error: 0.302500 | Prediction: 0.2500 | Delta: -0.2750 | New Weight: 0.7750
Iteration  4 | Error: 0.009587 | Prediction: 0.7021 | Delta: -0.0490 | New Weight: 0.8510
Iteration  8 | Error: 0.000304 | Prediction: 0.7826 | Delta: -0.0087 | New Weight: 0.9913
Iteration 12 | Error: 0.000010 | Prediction: 0.7969 | Delta: -0.0016 | New Weight: 1.0163
Iteration 16 | Error: 0.000000 | Prediction: 0.7994 | Delta: -0.0003 | New Weight: 1.0207
Iteration 19 | Error: 0.000000 | Prediction: 0.7998 | Delta: -0.0001 | New Weight: 1.0211

### 3.4 Breaking Gradient Descent (The Overshooting Phenomenon)
Let us demonstrate what happens when the input is large (`input = 2`) and no learning rate is applied ($\alpha = 1$):


In [1]:
weight = 0.5
goal_pred = 0.8
input_val = 2.0

print("--- Diverging Gradient Descent (Unscaled) ---")
for iteration in range(5):
    pred = input_val * weight
    error = (pred - goal_pred) ** 2
    delta = (pred - goal_pred) * input_val
    weight = weight - delta
    print(f"Iter {iteration}: Error={error:10.4f}, Pred={pred:8.4f}, Delta={delta:8.4f}, Weight={weight:8.4f}")

--- Diverging Gradient Descent (Unscaled) ---
Iter 0: Error=    0.0400, Pred=  1.0000, Delta=  0.4000, Weight=  0.1000
Iter 1: Error=    0.3600, Pred=  0.2000, Delta= -1.2000, Weight=  1.3000
Iter 2: Error=    3.2400, Pred=  2.6000, Delta=  3.6000, Weight= -2.3000
Iter 3: Error=   29.1600, Pred= -4.6000, Delta=-10.8000, Weight=  8.5000
Iter 4: Error=  262.4400, Pred= 17.0000, Delta= 32.4000, Weight=-23.9000

### 3.5 Taming Divergence with the Learning Rate ($\alpha$)
By scaling the update by $\alpha = 0.1$, the error stably converges to zero:


In [1]:
weight = 0.5
goal_pred = 0.8
input_val = 2.0
alpha = 0.1

print("--- Stabilized Gradient Descent with Alpha = 0.1 ---")
for iteration in range(20):
    pred = input_val * weight
    error = (pred - goal_pred) ** 2
    derivative = (pred - goal_pred) * input_val
    weight = weight - (alpha * derivative)
    if iteration % 4 == 0 or iteration == 19:
        print(f"Iter {iteration:2d} | Error: {error:.6f} | Pred: {pred:.4f} | Weight: {weight:.4f}")

--- Stabilized Gradient Descent with Alpha = 0.1 ---
Iter  0 | Error: 0.040000 | Pred: 1.0000 | Weight: 0.4600
Iter  4 | Error: 0.000518 | Pred: 0.8228 | Weight: 0.4036
Iter  8 | Error: 0.000007 | Pred: 0.8026 | Weight: 0.4004
Iter 12 | Error: 0.000000 | Pred: 0.8003 | Weight: 0.4000
Iter 16 | Error: 0.000000 | Pred: 0.8000 | Weight: 0.4000
Iter 19 | Error: 0.000000 | Pred: 0.8000 | Weight: 0.4000

---
## 4. Visualizing the Error Surface & Gradient Steps
The following Matplotlib diagram plots the quadratic error bowl $E(w) = (w \cdot x - y)^2$ and overlays the trajectory of gradient descent steps.


In [1]:
import matplotlib.pyplot as plt
import numpy as np

w_vals = np.linspace(-0.5, 2.5, 200)
x_fixed = 0.5
y_fixed = 0.8
errors = (w_vals * x_fixed - y_fixed) ** 2

# Track 5 iterations of gradient descent
w_track = [0.0]
for _ in range(5):
    p = x_fixed * w_track[-1]
    d = (p - y_fixed) * x_fixed
    w_track.append(w_track[-1] - d)

plt.figure(figsize=(9, 5))
plt.plot(w_vals, errors, 'navy', linewidth=2.5, label='Error Curve: $E(w) = (w \cdot 0.5 - 0.8)^2$')
e_track = [(w * x_fixed - y_fixed)**2 for w in w_track]

for i in range(len(w_track) - 1):
    plt.annotate('', xy=(w_track[i+1], e_track[i+1]), xytext=(w_track[i], e_track[i]),
                 arrowprops=dict(facecolor='crimson', edgecolor='crimson', width=1.5, headwidth=8))

plt.scatter(w_track, e_track, color='crimson', s=60, zorder=5, label='Gradient Descent Iterations')
plt.scatter([1.6], [0.0], color='darkgreen', marker='*', s=250, zorder=6, label='Global Minimum ($w^* = 1.6$)')

plt.title("Chapter 4: Gradient Descent Trajectory on Convex Error Bowl", fontsize=12)
plt.xlabel("Weight $w$")
plt.ylabel("Squared Error $E(w)$")
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why does the derivative term contain the input $x$?**  
   *A:* Because the chain rule yields $\frac{\partial E}{\partial w} = 2(\hat{y} - y) \frac{\partial \hat{y}}{\partial w}$, and since $\hat{y} = w x$, $\frac{\partial \hat{y}}{\partial w} = x$. Intuitively, the input acts as a transmission gear: a larger input scales the effect of weight changes on the output.

2. **Q: How does hot and cold learning scale as the number of network weights increases to $1,000$?**  
   *A:* It suffers from the curse of dimensionality. Testing all binary combinations of stepping higher or lower across $1,000$ weights requires $2^{1,000} \approx 10^{301}$ error evaluations per step, which is physically impossible. Gradient descent computes all $1,000$ partial derivatives simultaneously in a single backward pass.

3. **Q: What is the exact mathematical condition that causes gradient descent divergence?**  
   *A:* Divergence occurs when the effective step multiplier $|1 - 2\alpha x^2| > 1$, which happens whenever the learning rate exceeds $\alpha > \frac{1}{x^2}$.
